## Create QA Pairs

Run after the figures exist (`single/create_figures_batch.py` or
`create_figures_batch.py`), which writes `imgs/` + `jsons/` into a run directory.
This reads each figure's json and writes a `*_qa.json` alongside it containing
the generated question/answer pairs under a `VQA` key.

**Scope:** this copy generates only

1. **full-figure questions** - panel count, plotting style, colormap, aspect
   ratio, titles, axis labels, tick labels, which plot types appear
2. **contour-panel questions** - image-vs-lines, min/max/median/mean on
   x/y/color, distribution of color and x/y

plus the plot-type-agnostic error-bar question. Histogram, scatter, line and
cross-panel questions are not generated - see `models/utils/README.md` for why
(cross-panel in particular *looks* figure-level but is scatter/line/histogram
only).

**Note on sky panels:** `image of the sky` panels get only the full-figure and
error-bar questions - there is no sky-specific question module yet.


In [ ]:
# Run directory written by the figure generator: expects <save_dir_in>/jsons/
save_dir_in = "~/Downloads/tmp/test5_big/"

# where the *_qa.json files go
save_dir_out_json = "~/Dropbox/WASP2026/VQA/qa_jsons/"

In [ ]:
import json
from glob import glob
import os
import sys
from copy import deepcopy
from importlib import reload
import numpy as np
from functools import partial

# This notebook lives at the repo root; the QA code lives in models/utils/.
# models/ has no __init__.py, so models.utils is picked up as a namespace
# package once the repo root is on sys.path.
REPO_ROOT = os.getcwd()
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import models.utils.figure_level_qa_utils
import models.utils.contour_plot_qa_utils
import models.utils.sky_plot_qa_utils
import models.utils.qa_dispatch
reload(models.utils.figure_level_qa_utils)
reload(models.utils.contour_plot_qa_utils)
reload(models.utils.sky_plot_qa_utils)
reload(models.utils.qa_dispatch)

from models.utils.plot_qa_utils import init_qa_pairs
from models.utils.figure_level_qa_utils import figure_level_qa
from models.utils.misc_data_utils import NumpyEncoder
# dispatchers (composition layer, lifted out of this notebook into models/utils/)
from models.utils.qa_dispatch import (plot_level_contour_qa, plot_level_sky_qa,
                                      plot_level_general_qa, STATS, LINE_LIST)
from models.utils.sky_plot_qa_utils import SKY_LINE_LIST

stats = STATS
line_list = LINE_LIST


In [ ]:
save_dir_in = os.path.expanduser(save_dir_in)
save_dir_out_json = os.path.expanduser(save_dir_out_json)

if not os.path.exists(save_dir_out_json):
    os.makedirs(save_dir_out_json, exist_ok=True)   # makedirs: the path is nested
    print('made:', save_dir_out_json)

In [152]:
files_in = glob(save_dir_in + 'jsons/*.json')
files_in[:3]

['/Users/jnaiman/Dropbox/jcdl_followup/synthetic_figures/jsons/Picture_000121.json',
 '/Users/jnaiman/Dropbox/jcdl_followup/synthetic_figures/jsons/Picture_000064.json',
 '/Users/jnaiman/Dropbox/jcdl_followup/synthetic_figures/jsons/Picture_000176.json']

In [153]:
# get all plot types that have been used
plot_types = np.array([])
for fi in files_in:
    with open(fi, 'r') as f:
        data = json.load(f)
        data = json.loads(data)

    # overall panels
    for k,v in data.items():
        if 'plot' in k:
            plot_types = np.unique(np.concatenate([plot_types,[v['type']]]))
plot_types = plot_types.tolist()

plot_types


['contour', 'histogram', 'line', 'scatter']

In [ ]:
for fi in files_in:
    with open(fi, 'r') as f:
        data = json.load(f)
        data = json.loads(data)          # the figure jsons are double-encoded

    qa_pairs = init_qa_pairs()
    plot_nums = []
    for k, v in data.items():
        if 'plot' in k:
            plot_nums.append(int(k.split('plot')[-1]))

    # ---- (1) full-figure questions ----
    qa_pairs = figure_level_qa(data, deepcopy(qa_pairs), plot_types, verbose=False)

    # NOTE: cross_figure_qa() is intentionally not called. Both of its questions
    # are scatter/line/histogram only (calc_strongest skips contour explicitly),
    # so with those plot types excluded it would add nothing. See models/utils/README.md.

    # ---- per-panel questions ----
    for iplot in plot_nums:
        # leave off error bar gen
        # plot-type-agnostic (error bars), Level 2
        # qa_pairs = plot_level_general_qa(data, qa_pairs, iplot, verbose_qa=False)

        ptype = data['plot' + str(iplot)]['type']
        if ptype == 'contour':
            # ---- (2) contour-panel questions ----
            qa_pairs = plot_level_contour_qa(data, qa_pairs, iplot, stats,
                                             line_list=line_list, verbose_qa=False)
        elif ptype == 'image of the sky':
            # ---- (3) sky-panel questions ----
            # RA/DEC statistics come from the panel's WCS (xs/ys are pixel
            # indices, which do not match the RA/DEC printed on the axes).
            # The image-vs-lines question is off by default: the generator makes
            # it "image" ~99.8% of the time.
            qa_pairs = plot_level_sky_qa(data, qa_pairs, iplot, stats,
                                         line_list=SKY_LINE_LIST,
                                         ask_radec=True,
                                         ask_image_or_lines=False,
                                         verbose_qa=False)


    # update
    json_out = deepcopy(data)
    json_out['VQA'] = deepcopy(qa_pairs)
    dumped = json.dumps(json_out, cls=partial(NumpyEncoder, verbose=True))
    fiout = fi.split('/')[-1].removesuffix('.json') + '_qa.json'
    with open(save_dir_out_json + fiout, 'w') as f:
        json.dump(dumped, f)

print('!!!!! DONE !!!!')


In [163]:
qa_pairs['Level 3']

{'Plot-level questions': {'distribution-color + list)': {'plot0': {'Q': 'You are a helpful assistant that can analyze images.  Please choose the distribution from the following list: [random, linear, gaussian mixture model]. What is the underlying distribution used to create the data in this figure panel along the color-axis? Please format the output as a json as {"distribution color":""} for this figure panel, where the "distribution color" value should be a string, calculated from the data values used to create the plot.',
    'A': {'distribution + list)': 'random'},
    'note': 'this currently assumes all elements on a single plot have the same relationship type',
    'persona': 'You are a helpful assistant that can analyze images.',
    'context': ' Please choose the distribution from the following list: [random, linear, gaussian mixture model].',
    'question': 'What is the underlying distribution used to create the data in this figure panel along the color-axis?',
    'format': 

In [ ]:
# look at one
iPlot = 10

fi = save_dir_out_json + '/' + 'Picture_' + str(iPlot).zfill(6) + '_qa.json'
with open(fi, 'r') as f:
    data = json.load(f)
    data = json.loads(data)

In [162]:
data['VQA']

{'Level 1': {'Figure-level questions': {'rows/columns': {'Q': 'You are a helpful assistant that can analyze images.  How many panels are in this figure? Please format the output as a json as {"nrows":"", "ncols":""} to store the number of rows and columns.',
    'A': {'nrows': 2, 'ncols': 2},
    'persona': 'You are a helpful assistant that can analyze images.',
    'context': '',
    'question': 'How many panels are in this figure?',
    'format': 'Please format the output as a json as {"nrows":"", "ncols":""} to store the number of rows and columns.'},
   'plot style': {'Q': 'You are a helpful assistant that can analyze images. Assume this is a figure made with matplotlib in Python.  Examples of plotting styles are "classic" or "ggplot". Examples of plotting styles are "classic" or "ggplot". What is the plot style used in this figure? Please format the output as a json as {"plot style":""} to store the matplotlib plotting style used in the figure.',
    'A': {'plot style': 'seaborn-v